In [2]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [4]:
message="Tell me something about transformer technology"
response = ollama.chat(model='llama3.2', messages=[{"role":"user", "content":message}])
print(response['message']['content'])

Transformer technology has a rich history and plays a crucial role in modern electrical power systems. Here's an overview:

**What is a Transformer?**

A transformer is an electrical device that transfers electrical energy from one circuit to another through electromagnetic induction. It consists of two coils, known as the primary and secondary coils, which are wrapped around a common core.

**How Does it Work?**

When an alternating current (AC) flows through the primary coil, it generates a magnetic field. This magnetic field induces an electromotive force (EMF) in the secondary coil, causing a voltage to be induced across its ends. The transformer can step up or step down the voltage by adjusting the ratio of turns between the primary and secondary coils.

**Types of Transformers**

1. **Step-up Transformer**: Increases the voltage from a lower value to a higher value.
2. **Step-down Transformer**: Decreases the voltage from a higher value to a lower value.
3. **Isolation Transforme

In [5]:

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


class Website:
    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input", "link"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)


In [6]:
web= Website("https://pl.wikipedia.org/wiki/Du%C5%BCy_model_j%C4%99yzykowy")
print("Title: ", web.title)
print("text: ", web.text)

Title:  Duży model jęyzykowy – Wikipedia, wolna encyklopedia
text:  Przejdź do zawartości
Menu główne
Menu główne
przypnij
ukryj
Nawigacja
Strona główna
Losuj artykuł
Kategorie artykułów
Najlepsze artykuły
Częste pytania (FAQ)
Dla czytelników
O Wikipedii
Kontakt
Dla wikipedystów
Pierwsze kroki
Portal wikipedystów
Ogłoszenia
Zasady
Pomoc
Strony specjalne
Ostatnie zmiany
Szukaj
Szukaj
Wygląd
Wspomóż Wikipedię
Utwórz konto
Zaloguj się
Narzędzia osobiste
Wspomóż Wikipedię
Utwórz konto
Zaloguj się
Duży model jęyzykowy
Dodaj języki
Treść strony nie jest dostępna w innych językach.
Artykuł
Dyskusja
polski
Utwórz
Utwórz kod źródłowy
Narzędzia
Narzędzia
przypnij
ukryj
Działania
Utwórz
Utwórz kod źródłowy
Ogólne
Linkujące
Prześlij plik
Wersja do druku
Informacje o tej stronie
Zobacz skrócony adres URL
W innych projektach
Wygląd
przypnij
ukryj
Z Wikipedii, wolnej encyklopedii
W Wikipedii nie ma jeszcze artykułu o takiej nazwie. Możesz:
utworzyć go
,
zaproponować, żeby inni go napisali
,
poszukać 

In [8]:
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in polish."


In [9]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt


In [10]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]


In [11]:
messages_for(web)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in polish.'},
 {'role': 'user',
  'content': 'You are looking at a website titled Duży model jęyzykowy – Wikipedia, wolna encyklopedia\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\nPrzejdź do zawartości\nMenu główne\nMenu główne\nprzypnij\nukryj\nNawigacja\nStrona główna\nLosuj artykuł\nKategorie artykułów\nNajlepsze artykuły\nCzęste pytania (FAQ)\nDla czytelników\nO Wikipedii\nKontakt\nDla wikipedystów\nPierwsze kroki\nPortal wikipedystów\nOgłoszenia\nZasady\nPomoc\nStrony specjalne\nOstatnie zmiany\nSzukaj\nSzukaj\nWygląd\nWspomóż Wikipedię\nUtwórz konto\nZaloguj się\nNarzędzia osobiste\nWspomóż Wikipedię\nUtwórz konto\nZaloguj się\nDuży model jęyzykowy\nDodaj języki\nTreść strony n

In [14]:
MODEL='llama3.2'
def summarize(url):
    website = Website(url)
    response = ollama.chat(
        model=MODEL,
        messages=messages_for(website)
    )
    return response['message']['content']

In [16]:
summarize("https://wp.pl")

'"Wirtualna Polska Media Spółka Akcyjna" i "Wirtualna Polska Media S.A."'

In [17]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))
    

In [21]:
display_summary("https://wp.pl")

"Łucznik" objęty SAFE.  Posłowie PiS weszli do fabryki z kontrolą

In [33]:
system_prompt = """
Jesteś asystentem analizującym treść strony internetowej.
Twoim zadaniem jest wyciąganie najważniejszych informacji z dostarczonego tekstu.
Ignoruj elementy nawigacyjne, menu, stopki, linki, reklamy i treści niezwiązane z artykułem.
Skup się wyłącznie na kluczowych faktach, datach, osiągnięciach i informacjach merytorycznych.

Zawsze twórz:
- krótkie streszczenie,
- listę najważniejszych punktów,
- najważniejsze daty i wydarzenia,
- kluczowe osiągnięcia,
- jeśli to biografia — chronologię.

Odpowiadaj wyłącznie po polsku, w formie uporządkowanej i czytelnej.
"""


In [34]:
user_prompt = """
Przeanalizuj poniższą stronę i wyciągnij najważniejsze informacje.
Skup się na faktach, datach, osiągnięciach i kluczowych wydarzeniach.
Odpowiadaj po polsku.

URL:
https://pl.wikipedia.org/wiki/Kamil_Stoch
"""


In [35]:
models_to_test = [
    "qwen2.5:1.5b",
    "mistral",
    "llama3.2"
]

def test_models_with_prompt(prompt, models):
    results = {}
    for model in models:
        try:
            response = ollama.chat(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt}
                ]
            )
            results[model] = response['message']['content']
        except Exception as e:
            results[model] = f"Błąd modelu: {e}"
    return results

results = test_models_with_prompt(user_prompt, models_to_test)


In [36]:
def display_model_results(results):
    for model, answer in results.items():
        print(f"## Model: {model}\n")
        display(Markdown(answer))
        print("\n---\n")


In [37]:
display_model_results(results)


## Model: qwen2.5:1.5b



Ponieważ nie ma dostępu do treści strony internetowej, nie mogę skupić się na jej treść ani podać najważniejsze informacje. Aby analizować stronę, muszę przeglądnąć tekst.


---

## Model: mistral



 Krótkie streszczenie:
Kamil Stoch (ur. 19 marca 1988 w Zakopanem) – polski skoczek narciarski, zawodnik klubu AZS AWF Katowice, dwukrotny mistrz olimpijski (2014 i 2018), trzykrotny zdobywca Pucharu Świata (sezony 2013/2014, 2015/2016, 2017/2018) oraz dwukrotny mistrz świata (2013 i 2019).

Lista najważniejszych punktów:
- Urodzenie: 19 marca 1988 w Zakopanem.
- Klub sportowy: AZS AWF Katowice.
- Mistrzostwo olimpijskie w konkursach indywidualnych na skoczni normalnej i dużej (2014).
- Mistrzostwo olimpijskie w drużynowym skokach narciarskich z kombinacją norweską (2018).
- Pięć medali mistrzostw świata (dwa złote, dwa srebrne, jeden brązowy).
- Trzykrotny zdobywca Pucharu Świata (sezony 2013/2014, 2015/2016, 2017/2018).
- Dwukrotny zwycięzca Turnieju Czterech Skoczni (2013/2014 i 2014/2015).
- Złoty medal mistrzostw świata juniorów w drużynie (2007).
- Pierwszy w historii polski skoczek, który zdobył medale olimpijskie zarówno na normalnej jak i dużej skoczni.
- Wielokrotny zwycięzca konkursów Pucharu Świata (ponad 50).
- Srebrny medal mistrzostw świata w lotach narciarskich (2018).
- Mistrz Polski w skokach narciarskich (7 tytułów mistrzowskich).
- Rekordzista w liczbie wygranych konkursów Pucharu Świata na normalnej i dużej skoczni.

Najważniejsze daty i wydarzenia:
- 19 marca 1988 – urodzenie Kamil Stocha w Zakopanem.
- 2006 – debiut w Pucharze Świata.
- 3 lutego 2007 – złoty medal mistrzostw świata juniorów w drużynie.
- 12 grudnia 2009 – pierwsze zwycięstwo w Pucharze Świata (Wałkowe, Niemcy).
- 24 lutego 2013 – złoty medal mistrzostw świata na skoczni normalnej.
- 2 marca 2013 – srebrny medal mistrzostw świata w lotach narciarskich.
- 8 lutego 2014 – złoty medal olimpijski na skoczni dużej.
- 9 lutego 2014 – złoty medal olimpijski na skoczni normalnej.
- 3 lutego 2015 – drugi tytuł mistrza świata w lotach narciarskich.
- 20 grudnia 2015 – złoty medal Turnieju Czterech Skoczni (Innsbruck).
- 17 lutego 2018 – srebrny medal mistrzostw świata w lotach narciarskich.
- 18 lutego 2018 – złoty medal olimpijski w drużynowym skokach narciarskich z kombinacją norweską.
- 24 lutego 2018 – brązowy medal olimpijski na skoczni dużej.
- 25 lutego 2018 – srebrny medal olimpijski na skoczni normalnej.

Kluczowe osiągnięcia:
- Pierwszy w historii polski skoczek, który zdobył medale olimpijskie zarówno na normalnej jak i dużej skoczni (2014).
- Pierwszy w historii polski skoczek, który wygrał Turniej Czterech Skoczni dwukrotnie (w sezonach 2013/2014 i 2014/2015).
- Rekordzista w liczbie wygranych konkursów Pucharu Świata na normalnej skoczni (ponad 30).
- Rekordzista w liczbie wygranych konkursów Pucharu Świata na dużej skoczni (ponad 20).
- Pierwszy skoczek w historii, który zaliczył ponad 50 zwycięstw w Pucharze Świata.
- Jeden z nielicznych sportowców, którzy zdobyli trzykrotnie Puchar Świata (2013/2014, 2015/2016, 2017/2018).

Chronologia biografii:
- 19 marca 1988 – urodzenie w Zakopanem.
- 2006 – debiut w Pucharze Świata.
- 2007 – mistrzostwo świata juniorów w drużynie (Zakopane).
- 2009 – pierwsze zwycięstwo w Pucharze Świata (Wałkowe, Niemcy).
- 2013 – złoty medal mistrzostw świata na skoczni normalnej oraz srebrny medal mistrzostw świata w lotach narciarskich.
- 2014 – dwa medale olimpijskie (na skoczni dużej i normalnej).
- 2015 – zwycięstwo w Turnieju Czterech Skoczni oraz srebrny medal mistrzostw świata w lotach narciarskich.
- 2016 – trzecie miejsce w klasyfikacji generalnej Pucharu Świata.
- 2017 – trzeci tytuł mistrza świata w lotach narciarskich.
- 2018 – dwa medale olimpijskie (na skoczni dużej i normalnej) oraz złoty medal olimpijski w drużynowym skokach narciarskich z kombinacją norweską.


---

## Model: llama3.2



**Streszczenie:**
Kamil Stoch to polski skoczek narciarski, który reprezentuje Polskę na międzynarodowych zawodach. Wystartował w Pucharze Świata, Pucharze Kontynentalnym oraz w meczach drużynowych.

**Najważniejsze punkty:**

* Kamil Stoch jest polskim skocznikiem narciarskim.
* W swojej karierze zwyciężył w numerouszach między innymi w Pucharze Świata, Pucharze Kontynentalnym oraz na Meczach Drużynowych.
* Jego najlepszym osiągnięciem jest trzeci tytuł Mistrza Świata w skokach narciarskich.
* Reprezentuje Polskę na międzynarodowych zawodach.

**Najważniejsze daty i wydarzenia:**

* 13 lutego 2015: Zwycięstwo Kamil Stocha w Pucharze Świata.
* 24 stycznia 2016: Zwycięstwo Kamil Stocha w meczu drużynowym.
* 16 marca 2018: Trzeci tytuł Mistrza Świata w skokach narciarskich.

**Kluczowe osiągnięcia:**

* Zwycięstwa w Pucharze Świata, Pucharze Kontynentalnym oraz na Mezzu Drużynowych.
* Trzeci tytuł Mistrza Świata w skokach narciarskich.

**Chronologia biografii:**
Kamil Stoch urodził się 11 lipca 1990 r. w Zakopanem, Polska. Zaczynał karierę skocznictwa w 2006 roku. W latach 2014-2022 reprezentował Polskę na mistrzostwach świata i mistrzostwach Europy w skokach narciarskich.


---

